In [1]:
import pandas as pd
import re

import string
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('CaptstoneProjectData_2025.csv')

# Observe data

In [3]:
df.head()

,Subject,Body,Unnamed: 2,Unnamed: 3
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,NaN,NaN
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,NaN,NaN
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,NaN,NaN
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,NaN,NaN
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Subject     2467 non-null   object 
 1   Body        2571 non-null   object 
 2   Unnamed: 2  0 non-null      float64
 3   Unnamed: 3  0 non-null      float64
dtypes: float64(2), object(2)
memory usage: 80.6+ KB


# Deal with missing/duplicate data

In [5]:
# Remove columns with all null values
valid_df = df.dropna(axis=1, how='all')
valid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Subject  2467 non-null   object
 1   Body     2571 non-null   object
dtypes: object(2)
memory usage: 40.4+ KB


In [6]:
# Check for duplicate rows
valid_df.duplicated().any()

np.True_

In [7]:
# Drop duplicate records
valid_df = valid_df.drop_duplicates()
valid_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2351 entries, 0 to 2575
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Subject  2250 non-null   object
 1   Body     2347 non-null   object
dtypes: object(2)
memory usage: 55.1+ KB


In [8]:
# Check for missing values (NaNs)
valid_df.isna().sum()

,0
Subject,101
Body,4


In [9]:
# Impute missing values
valid_df['Subject'] = valid_df['Subject'].fillna('')
valid_df['Body'] = valid_df['Body'].fillna('')
valid_df['Subject_is_empty'] = valid_df['Subject'] == ''
valid_df['Body_is_empty'] = valid_df['Body'] == ''

In this step, missing values in the Subject and Body columns are imputed using empty strings (''). This ensures that all entries are string-type and avoids errors during text processing or vectorization. Instead of dropping the rows, we preserve them, as empty subjects or bodies may indicate phishing patterns.

Additionally, two new boolean columns, Subject_is_empty and Body_is_empty, are created to explicitly mark which entries were originally missing or empty. These flags can serve as useful features during pattern recognition or machine learning, as phishing emails often have missing or blank content fields.

In [10]:
valid_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2351 entries, 0 to 2575
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Subject           2351 non-null   object
 1   Body              2351 non-null   object
 2   Subject_is_empty  2351 non-null   bool  
 3   Body_is_empty     2351 non-null   bool  
dtypes: bool(2), object(2)
memory usage: 59.7+ KB


In [11]:
valid_df.head(10)

,Subject,Body,Subject_is_empty,Body_is_empty
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,False,False
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,False,False
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,False,False
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,False,False
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,False,False
5,Account validation for uvic.ca user(s),"Hi helpdesk,\n\nTo continue using your email a...",False,False
6,Y‌our p‌ayme‌nt m‌et‌hod h‌as b‌een Dec‌li‌ned...,Notice: This message was sent from outside the...,False,False
7,Microsoft account security notification,Notice: This message was sent from outside the...,False,False
8,Urgent :AutoPay Payment was Unsuccessful !,[https://www.telstra.com.au/content/dam/tcom/a...,False,False
9,Verify your identity,Notice: This message was sent from outside the...,False,False


# Language normalization

Since we saw some non-english text in Subject and Body, we use langdetect to identify the language of the subject and body seperately

In [12]:
! pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=89e1e97ed6364e1faaefa3fb95a2eea420408cc11beaea7d8416ec16d2d5ee5b
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [13]:
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

In [14]:
extract_df = valid_df.copy()
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

extract_df['subject_lang'] = extract_df['Subject'].apply(detect_language)
extract_df['body_lang'] = extract_df['Body'].apply(detect_language)

In [15]:
extract_df.head()

,Subject,Body,Subject_is_empty,Body_is_empty,subject_lang,body_lang
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,False,False,fr,en
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,False,False,ru,et
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,False,False,en,en
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,False,False,en,en
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,False,False,en,en


use googletrans to translate non-English to English

In [16]:
! pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.6 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=dc534e9d382aa713da0496addfd6861e8f67452c74145408e0dfa07b7d0f055d
  Stored in directory: /root/.cache/pip/wheels/39/17/6f/66a045ea3d168826074691b4b787b8f324d3f646d755443fda
Successfully built googletrans
  Attempting uninstall: hyperframe
    Found existing installation: hyperframe 6.1.0
    Uninstalling hyperfra

In [17]:
from googletrans import Translator

In [18]:
translator = Translator()

def translate_if_not_english(text, lang_code):
    try:
        if lang_code == 'en':
            return text
        result = translator.translate(text, dest='en')
        return result.text
    except:
        return text  # fallback to original

extract_df['subject_en'] = extract_df.apply(
    lambda row: translate_if_not_english(row['Subject'], row['subject_lang']), axis=1
)

extract_df['body_en'] = extract_df.apply(
    lambda row: translate_if_not_english(row['Body'], row['body_lang']), axis=1
)


In [19]:
extract_df.head()

,Subject,Body,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,False,False,fr,en,®REVIEW your Shipment Details / Shipment Notif...,Notice: This message was sent from outside the...
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,False,False,ru,et,Our Assound I-ON HOLD,Votre réponse a bien été prise en compte.\n[ht...
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,False,False,en,en,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,False,False,en,en,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,False,False,en,en,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...


In [20]:
extract_df_1 = extract_df.copy()

# Feature Extraction

In [21]:
extract_df_2 = extract_df.copy()

In [67]:
import re

def extract_emails(text):
    if isinstance(text, str):
        return re.findall(r'[\w\.-]+@[\w\.-]+', text)
    return []

def extract_attachment_keywords(text):
    if isinstance(text, str):
        return re.findall(r'\b(attachment|attached|pdf|docx|xls)\b', text, re.IGNORECASE)
    return []

def extract_special_characters(text):
    if isinstance(text, str):
        return re.findall(r'[!@#$%^&*()_+=\[\]{}|\\;:\'",<>/?`~]', text)
    return []

def extract_urls(text):
    if isinstance(text, str):
        return re.findall(r'(https?://\S+)', text)
    return []

def remove_extracted(text, extracted_items):
    if not isinstance(text, str):
        return ''
    for item in extracted_items:
        text = text.replace(item, '')
    return text.strip()

# Subject extractions
# extract_df_2['subject_links'] = extract_df_2['subject_en'].apply(extract_urls)
extract_df_2['subject_emails'] = extract_df_2['subject_en'].apply(extract_emails)
extract_df_2['subject_attachments'] = extract_df_2['subject_en'].apply(extract_attachment_keywords)
extract_df_2['subject_special_chars'] = extract_df_2['subject_en'].apply(extract_special_characters)

extract_df_2['remained_subject_en'] = extract_df_2.apply(
    lambda row: remove_extracted(
        row['subject_en'],
        row['subject_emails'] + row['subject_attachments'] + row['subject_special_chars']
    ),
    axis=1
)

# Body extractions
extract_df_2['body_links'] = extract_df_2['body_en'].apply(extract_urls)
extract_df_2['body_emails'] = extract_df_2['body_en'].apply(extract_emails)
extract_df_2['body_attachments'] = extract_df_2['body_en'].apply(extract_attachment_keywords)
extract_df_2['body_special_chars'] = extract_df_2['body_en'].apply(extract_special_characters)

extract_df_2['remained_body_en'] = extract_df_2.apply(
    lambda row: remove_extracted(
        row['body_en'],
        row['body_links'] + row['body_emails'] + row['body_attachments'] + row['body_special_chars']
    ),
    axis=1
)
extract_df_2['remained_body_en'] = extract_df_2['remained_body_en'].str.replace(r'[\r\n]+', ' ', regex=True)



In [23]:
extract_df_2.head(10)

,Subject,Body,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en,subject_emails,subject_attachments,subject_special_chars,remained_subject_en,body_links,body_emails,body_attachments,body_special_chars,remained_body_en
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,False,False,fr,en,®REVIEW your Shipment Details / Shipment Notif...,Notice: This message was sent from outside the...,[],[],"[®, /, /, :]",REVIEW your Shipment Details Shipment Notific...,[https://www.canadapost-postescanada.ca/cpc/as...,"[amuench@uvic.ca, hudsonesajoyce@gmail.com]",[],"[:, ., ., [, :, /, /, ., -, ., /, /, /, /, /, ...",Notice This message was sent from outside the ...
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,False,False,ru,et,Our Assound I-ON HOLD,Votre réponse a bien été prise en compte.\n[ht...,[],[],[-],Our Assound ION HOLD,[https://www.edigitalagence.com.au/wp-connet/u...,[foipp@uvic.ca],[netflix-logo-red-black-png.png],"[., [, :, /, /, ., ., ., /, -, /, /, -, -, -, ...",Votre réponse a bien été prise en compte Υоur ...
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,False,False,en,en,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,[],[],"[:, #, -, ., +, ., ., .]",Completed Invoice KZ89TYS2564 fromBestbuycom ...,[https://NA4.docusign.net/member/Images/email/...,"[auwaluu.ma.r.bu.ba@googlemail.com, icon-Downl...","[protect.doc, docComplete-white.png, www.doc, ...","[:, ., ., [, ], [, :, /, /, ., ., /, /, /, /, ...",Notice This message was sent from outside the ...
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,False,False,en,en,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,[],[],[!],UVic IMPORTANT NOTICE,[https://forms.gle/TpBxJ1SRFwgYMd8c7>],[],[],"[/, ., ., ., <, :, /, /, ., /, >, -, ., ., ,]",Your UVIC account has been filed under the lis...
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,False,False,en,en,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,[],[],"[(, )]",You have 6 Suspended incoming messages,[https://googleweblight.com/i?u=https://cloudf...,"[helpdesk@uvic.ca, helpdesk@uvic.ca]",[],"[., ., ., :, @, ., (, ), <, :, /, /, ., /, ?, ...",Message generated from uvicca source Sender ...
5,Account validation for uvic.ca user(s),"Hi helpdesk,\n\nTo continue using your email a...",False,False,en,en,Account validation for uvic.ca user(s),"Hi helpdesk,\n\nTo continue using your email a...",[],[],"[., (, )]",Account validation for uvicca users,[https://apiservices.krxd.net/click_tracker/tr...,[helpdesk@uvic.ca],[],"[,, (, @, ., ), ,, ., <, :, /, /, ., ., /, /, ...",Hi helpdesk To continue using your email accou...
6,Y‌our p‌ayme‌nt m‌et‌hod h‌as b‌een Dec‌li‌ned...,Notice: This message was sent from outside the...,False,False,en,en,Y‌our p‌ayme‌nt m‌et‌hod h‌as b‌een Dec‌li‌ned...,Notice: This message was sent from outside the...,[],[],"[‌, ‌, ‌, ‌, ‌, ‌, ‌, ‌, ‌, ., ‌, #, ‌, -]",Your payment method has been Declined CaseID 9...,[https://script.google.com/macros/s/AKfycbwUfF...,[],[],"[:, ., ., ., ., :, *, *, ., ., ., <, :, /, /, ...",Notice This message was sent from outside the ...
7,Microsoft account security notification,Notice: This message was sent from outside the...,False,False,en,en,Microsoft account security notification,Notice: This message was sent from outside the...,[],[],[],Microsoft account security notification,[https://go.microsoft.com/fwlink/?LinkId=20867...,"[a@hotmail.com, a@hotmail.com]",[],"[:, ., ., /, /, ,, *, *, @, ., <, :, *, *, @, ...",Notice This message was sent from outside the ...
8,Urgent :AutoPay Payment was Unsuccessful !,[https://www.telstra.com.au/content/dam/tcom/a...,False,False,en,en,Urgent :AutoPay Payment was Unsuccessful !,[https://www.telstra.com

Note: there are emails and attachments in Subject, while there is no links in Subject.

In [24]:
extract_df_2.loc[extract_df_2['subject_emails'].apply(lambda x: len(x) > 0), 'subject_emails']

,subject_emails
77,[helpdesk@uvic.ca]
324,[drsudhirruparelia@gmail.com]
356,[helpdesk@uvic.ca]
364,[vpac@uvic.ca]
365,[vpac@uvic.ca]
...,...
2434,[procurementofficer4@uvic.ca]
2475,[ljh@uvic.ca]
2491,[rpkmadmin@uvic.ca]
2516,[biocoop@uvic.ca]


In [25]:
extract_df_2.loc[extract_df_2['subject_attachments'].apply(lambda x: len(x) > 0), 'subject_attachments']

,subject_attachments
89,[accessment.docx]
93,[Updates.docx]
119,[Bonus.docx]
297,[Agreement.Pdf]
671,[Agreement.pdf]
770,[ihg_logo_folio7253703.pdf]
2296,[EVALUATION.docx]
2438,[EVALUATION.docx]
2537,[.1055043768.xls]


# Output result

In [26]:
extract_df_2.to_csv("cleaned_email_data.csv", index=False, encoding='utf-8-sig')

In [27]:
phishing_df_cleaned = pd.read_csv('cleaned_email_data.csv')
phishing_df_cleaned.head(5)

,Subject,Body,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en,subject_emails,subject_attachments,subject_special_chars,remained_subject_en,body_links,body_emails,body_attachments,body_special_chars,remained_body_en
0,®Review your shipment details / Shipment Notif...,Notice: This message was sent from outside the...,False,False,fr,en,®REVIEW your Shipment Details / Shipment Notif...,Notice: This message was sent from outside the...,[],[],"['®', '/', '/', ':']",REVIEW your Shipment Details Shipment Notific...,['https://www.canadapost-postescanada.ca/cpc/a...,"['amuench@uvic.ca', 'hudsonesajoyce@gmail.com']",[],"[':', '.', '.', '[', ':', '/', '/', '.', '-', ...",Notice This message was sent from outside the ...
1,Υоur ассоunt іѕ оn hоld,\nVotre réponse a bien été prise en compte.\n[...,False,False,ru,et,Our Assound I-ON HOLD,Votre réponse a bien été prise en compte.\n[ht...,[],[],['-'],Our Assound ION HOLD,['https://www.edigitalagence.com.au/wp-connet/...,['foipp@uvic.ca'],['netflix-logo-red-black-png.png'],"['.', '[', ':', '/', '/', '.', '.', '.', '/', ...",Votre réponse a bien été prise en compte Υоur ...
2,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,False,False,en,en,Completed: Invoice # KZ89TYS2564 from-Bestbuy....,Notice: This message was sent from outside the...,[],[],"[':', '#', '-', '.', '+', '.', '.', '.']",Completed Invoice KZ89TYS2564 fromBestbuycom ...,['https://NA4.docusign.net/member/Images/email...,"['auwaluu.ma.r.bu.ba@googlemail.com', 'icon-Do...","['protect.doc', 'docComplete-white.png', 'www....","[':', '.', '.', '[', ']', '[', ':', '/', '/', ...",Notice This message was sent from outside the ...
3,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,False,False,en,en,UVic IMPORTANT NOTICE!,Your UVIC account has been filed under the lis...,[],[],['!'],UVic IMPORTANT NOTICE,['https://forms.gle/TpBxJ1SRFwgYMd8c7>'],[],[],"['/', '.', '.', '.', '<', ':', '/', '/', '.', ...",Your UVIC account has been filed under the lis...
4,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,False,False,en,en,You have (6) Suspended incoming messages,\n\nMessage generated from uvic.ca source.\n\...,[],[],"['(', ')']",You have 6 Suspended incoming messages,['https://googleweblight.com/i?u=https://cloud...,"['helpdesk@uvic.ca', 'helpdesk@uvic.ca']",[],"['.', '.', '.', ':', '@', '.', '(', ')', '<', ...",Message generated from uvicca source Sender ...


In [53]:
enron_df = pd.read_csv('Enron.csv', nrows=5000)
enron_df.sample(5)

,subject,body,label
193,revision # 1 - enron / hpl actuals for sept . ...,teco tap 45 . 000 / enron ; 61 . 250 / hpl gas...,0
338,"hpl nom for july 1 , 2000",( see attached file : hplo 701 . xls )\r\n- hp...,0
2004,mobil beaumont,rebecca :\r\ni spoke with brian nichols about ...,0
4616,diet medications online,stop wasting money on prescription drugs . get...,1
1188,final guest list & rooming list,attached for your reference and review is the ...,0


In [54]:
enron_df.head(5)

,subject,body,label
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0


In [55]:
enron_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   subject  4960 non-null   object
 1   body     5000 non-null   object
 2   label    5000 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 117.3+ KB


In [56]:
valid_enron_df = enron_df.dropna(axis=1, how='all')
valid_enron_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   subject  4960 non-null   object
 1   body     5000 non-null   object
 2   label    5000 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 117.3+ KB


In [57]:
valid_enron_df.duplicated().any()

np.False_

In [58]:
valid_enron_df = valid_enron_df.drop_duplicates()
valid_enron_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   subject  4960 non-null   object
 1   body     5000 non-null   object
 2   label    5000 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 117.3+ KB


In [59]:
valid_enron_df.isna().sum()

,0
subject,40
body,0
label,0


In [60]:
valid_enron_df.sample(5)

,subject,body,label
3016,koch midstream services co,i have been billing koch midstream services co...,0
3757,30 seconds refinance,"hello ,\r\nwe sent you an email a while ago , ...",1
3102,cornhusker,daren - - - ( re : the email below from kathy ...,0
2442,hpl meter # 985355 brown common point,daren :\r\nduring the period of 1 / 1 / 99 thr...,0
1215,enron methanol,"for maro 0 , siatar # 139055 , you will need t...",0


In [61]:
valid_enron_df['Subject_is_empty'] = valid_enron_df['subject'] == ''
valid_enron_df['Body_is_empty'] = valid_enron_df['body'] == ''


In [62]:
valid_enron_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   subject           4960 non-null   object
 1   body              5000 non-null   object
 2   label             5000 non-null   int64 
 3   Subject_is_empty  5000 non-null   bool  
 4   Body_is_empty     5000 non-null   bool  
dtypes: bool(2), int64(1), object(2)
memory usage: 127.1+ KB


In [63]:
valid_enron_df['subject_lang'] = valid_enron_df['subject'].fillna('').apply(detect_language)
valid_enron_df['body_lang'] = valid_enron_df['body'].fillna('').apply(detect_language)

In [64]:
valid_enron_df['subject_en'] = valid_enron_df.apply(
    lambda row: translate_if_not_english(row['subject'], row['subject_lang']), axis=1
)

valid_enron_df['body_en'] = valid_enron_df.apply(
    lambda row: translate_if_not_english(row['body'], row['body_lang']), axis=1
)

In [65]:
valid_enron_df.head()

,subject,body,label,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0,False,False,en,en,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0,False,False,en,en,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0,False,False,en,en,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo..."
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0,False,False,en,en,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0,False,False,no,en,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...


In [68]:
valid_enron_df['subject_emails'] = valid_enron_df['subject_en'].apply(extract_emails)
valid_enron_df['subject_attachments'] = valid_enron_df['subject_en'].apply(extract_attachment_keywords)
valid_enron_df['subject_special_chars'] = valid_enron_df['subject_en'].apply(extract_special_characters)

valid_enron_df['remained_subject_en'] = valid_enron_df.apply(
    lambda row: remove_extracted(
        row['subject_en'],
        row['subject_emails'] + row['subject_attachments'] + row['subject_special_chars']
    ),
    axis=1
)

# Body extractions
valid_enron_df['body_links'] = valid_enron_df['body_en'].apply(extract_urls)
valid_enron_df['body_emails'] = valid_enron_df['body_en'].apply(extract_emails)
valid_enron_df['body_attachments'] = valid_enron_df['body_en'].apply(extract_attachment_keywords)
valid_enron_df['body_special_chars'] = valid_enron_df['body_en'].apply(extract_special_characters)

valid_enron_df['remained_body_en'] = valid_enron_df.apply(
    lambda row: remove_extracted(
        row['body_en'],
        row['body_links'] + row['body_emails'] + row['body_attachments'] + row['body_special_chars']
    ),
    axis=1
)
valid_enron_df['remained_body_en'] = valid_enron_df['remained_body_en'].str.replace(r'[\r\n]+', ' ', regex=True)

In [69]:
valid_enron_df.head(5)

,subject,body,label,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en,subject_emails,subject_attachments,subject_special_chars,remained_subject_en,body_links,body_emails,body_attachments,body_special_chars,remained_body_en
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0,False,False,en,en,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,[],[],"[,]",hpl nom for may 25 2001,[],[],"[attached, xls, xls]","[(, :, )]",see file hplno 525 . - hplno 525 .
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0,False,False,en,en,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,[],[],"[:, /]",re nom actual vols for 24 th,[],[],[],"[/, /, /, /, :, &, :, "", "", /, /, :, :, @, ,, ...",- - - - - - - - - - - - - - - - - - - - - - fo...
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0,False,False,en,en,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",[],[],"[,]",enron actuals for march 30 - april 1 201,[],[],[],"[,, ,, ,]",estimated actuals march 30 2001 no flow march...
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0,False,False,en,en,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,[],[],"[,]",hpl nom for may 30 2001,[],[],"[attached, xls, xls]","[(, :, )]",see file hplno 530 . - hplno 530 .
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0,False,False,no,en,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,[],[],"[,]",hpl nom for june 1 2001,[],[],"[attached, xls, xls]","[(, :, )]",see file hplno 601 . - hplno 601 .


In [70]:
clean_emails_df = valid_enron_df
phishing_emails_df = phishing_df_cleaned

In [71]:
phishing_emails_df['label'] = 1

In [72]:
final_df_combined = pd.concat([clean_emails_df, phishing_emails_df], ignore_index=True)

final_df_combined = final_df_combined.sample(frac=1, random_state=42).reset_index(drop=True)


In [73]:
final_df_combined.sample(20)

,subject,body,label,Subject_is_empty,Body_is_empty,subject_lang,body_lang,subject_en,body_en,subject_emails,subject_attachments,subject_special_chars,remained_subject_en,body_links,body_emails,body_attachments,body_special_chars,remained_body_en,Subject,Body
2853,new financial operations contacts for energy o...,i just wanted to give all of you an update to ...,0,False,False,en,en,new financial operations contacts for energy o...,i just wanted to give all of you an update to ...,[],[],[],new financial operations contacts for energy o...,[],[],[],"[;, ,, ,, ,, ,, ,]",i just wanted to give all of you an update to ...,NaN,NaN
5731,best free adult dating,search for sexual partners in your areaclick h...,1,False,False,en,en,best free adult dating,search for sexual partners in your areaclick h...,[],[],[],best free adult dating,[],[],[],[!],search for sexual partners in your areaclick h...,NaN,NaN
4181,i ' ll never stop loving you,at what age is it easiest to fall in love . ?\...,1,False,False,no,en,i ' ll never stop loving you,at what age is it easiest to fall in love . ?\...,[],[],['],i ll never stop loving you,[],[],[],"[?, :, :, /, /, /, /, ?, =]",at what age is it easiest to fall in love . i...,NaN,NaN
1267,tom o ' connor shut - in list,the following meters were shut - in for approx...,0,False,False,en,en,tom o ' connor shut - in list,the following meters were shut - in for approx...,[],[],['],tom o connor shut - in list,[],[],[],"[:, ,, ', ', #, #, #, #, ']",the following meters were shut - in for approx...,NaN,NaN
731,re :,- - - - - - - - - - - - - - - - - - - - - - fo...,0,False,False,ro,en,re :,- - - - - - - - - - - - - - - - - - - - - - fo...,[],[],[:],re,[],[],[attached],"[/, /, /, /, :, @, /, /, :, :, :, @, :, :, :, ...",- - - - - - - - - - - - - - - - - - - - - - fo...,NaN,NaN
4199,NaN,NaN,1,False,False,fr,en,Payment Invoice (IN09675),Notice: This message was sent from outside the...,[],[],"['(', ')']",Payment Invoice IN09675,[],[],[],"[':', '.', '.', ',', '/', '/', '.', '.', '.', ...",Notice This message was sent from outside the ...,Payment Invoice (IN09675),Notice: This message was sent from outside the...
5301,look through the message - manager,do you wish to become multi - orgasmic ? it ' ...,1,False,False,en,en,look through the message - manager,do you wish to become multi - orgasmic ? it ' ...,[],[],[],look through the message - manager,[],[],[],"[?, ', !, !, !, ?, :, :, /, /, /, ?]",do you wish to become multi - orgasmic it s ...,NaN,NaN
2449,"hpl nomination for january 26 , 2000",( see attached file : hplol 26 . xls )\r\n- hp...,0,False,False,en,en,"hpl nomination for january 26 , 2000",( see attached file : hplol 26 . xls )\r\n- hp...,[],[],"[,]",hpl nomination for january 26 2000,[],[],"[attached, xls, xls]","[(, :, )]",see file hplol 26 . - hplol 26 .,NaN,NaN
783,entex apr 3 noms,- - - - - - - - - - - - - - - - - - - - - - fo...,0,False,False,fr,en,entex apr 3 noms,- - - - - - - - - - - - - - - - - - - - - - fo...,[],[],[],entex apr 3 noms,[],[],"[xls, attachment, xls, xls]","[/, /, /, /, :, _, _, @, /, /, :, :, :, @, ,, ...",- - - - - - - - - - - - - - - - - - - - - - fo...,NaN,NaN
3572,we have all your favorite programs at incredib...,windows x . p professi 0 nal update\r\nwe migh...,1,False,False,en,en,we have all your favorite programs at incredib...,windows x . p professi 0 nal update\r\nwe migh...,[],[],[],we have all your favorite programs at incredib...,[],[],[],"[:, :, :, /, /, /, ?, :, /, ,, ,, ,, :, :, :, ...",windows x . p professi 0 nal update we might h...,NaN,NaN


### MODEL EXPERIMENTATION

In [74]:
# Convert list features to counts
final_df_combined['subject_special_char_count'] = final_df_combined['subject_special_chars'].apply(lambda x: len(x) if isinstance(x, list) else 0)
final_df_combined['body_special_char_count'] = final_df_combined['body_special_chars'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Need to figure out how to count number of links in the body
#final_df_combined['body_link_count'] = final_df_combined['body_links'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Clean and convert boolean-like features
for col in ['Subject_is_empty', 'Body_is_empty', 'subject_en', 'body_en']:
    final_df_combined[col] = final_df_combined[col].apply(lambda x: bool(x) if isinstance(x, (bool, int)) else False).astype(int)

# Encode language fields safely
le_subject = LabelEncoder()
le_body = LabelEncoder()
final_df_combined['subject_lang_enc'] = le_subject.fit_transform(final_df_combined['subject_lang'].fillna('unknown').astype(str))
final_df_combined['body_lang_enc'] = le_body.fit_transform(final_df_combined['body_lang'].fillna('unknown').astype(str))

# Optional: Add text length features
final_df_combined['subject_len'] = final_df_combined['remained_subject_en'].fillna('').astype(str).apply(len)
final_df_combined['body_len'] = final_df_combined['remained_body_en'].fillna('').astype(str).apply(len)

In [75]:
# Select features
feature_cols = [
    'Subject_is_empty', 'Body_is_empty', 'subject_en', 'body_en',
    'subject_special_char_count', 'body_special_char_count',
    'subject_lang_enc', 'body_lang_enc',
    'subject_len', 'body_len'
]

X = final_df_combined[feature_cols]
y = final_df_combined['label']


In [76]:
X.head(20)

,Subject_is_empty,Body_is_empty,subject_en,body_en,subject_special_char_count,body_special_char_count,subject_lang_enc,body_lang_enc,subject_len,body_len
0,0,0,0,0,0,0,7,7,8,291
1,0,0,0,0,1,106,14,7,30,1796
2,0,0,0,0,0,0,10,7,17,147
3,0,0,0,0,0,26,6,7,45,1124
4,0,0,0,0,0,0,6,7,30,635
5,0,0,0,0,0,0,10,7,29,948
6,0,0,0,0,0,2,1,7,24,230
7,0,0,0,0,3,43,6,7,49,552
8,0,0,0,0,0,0,6,7,57,492
9,0,0,0,0,0,0,6,7,29,1045


In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [78]:
# Train LGBM
clf = lgb.LGBMClassifier(verbose=-1)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.93      0.86       730
           1       0.92      0.78      0.85       741

    accuracy                           0.86      1471
   macro avg       0.86      0.86      0.86      1471
weighted avg       0.86      0.86      0.86      1471

